# S6E7: CatBoost V2-Core Diversity Submission

E020'nin Kaggle NVIDIA GPU üzerinde çalışabilen notebook sürümü: fold-safe V2-Core features + 3-fold CatBoost + sqrt-balanced sample weights + OOF multiplier tuning.

Bu notebook E002'nin yerini almak için değil, önce CatBoost standalone LB korelasyonunu hızlı ölçmek ve sonrasında E022/E023 ensemble'larına probability kaynağı üretmek için tasarlanmıştır.


## 1. Setup

In [ ]:
from pathlib import Path
import gc
import json
import platform
import time
import warnings

import catboost
from catboost import CatBoostClassifier
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, recall_score
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid", context="notebook")

SEED = 42
DRY_RUN = True  # Set False for the real 3-fold Kaggle submission run.
DRY_TRAIN_ROWS = 12_000
DRY_TEST_ROWS = 4_000
ID_COL = "id"
TARGET = "health_condition"
CLASSES = ["at-risk", "fit", "unhealthy"]
CLASS_TO_ID = {name: i for i, name in enumerate(CLASSES)}
NUM_COLS = [
    "sleep_duration", "heart_rate", "bmi", "calorie_expenditure",
    "step_count", "exercise_duration", "water_intake",
]
CAT_COLS = [
    "diet_type", "stress_level", "sleep_quality", "physical_activity_level",
    "smoking_alcohol", "gender",
]
RATIO_DEFS = {
    "calorie_per_step": ("calorie_expenditure", "step_count"),
    "calorie_per_exercise_min": ("calorie_expenditure", "exercise_duration"),
    "step_per_exercise_min": ("step_count", "exercise_duration"),
    "water_per_bmi": ("water_intake", "bmi"),
    "exercise_per_bmi": ("exercise_duration", "bmi"),
    "steps_per_sleep_hour": ("step_count", "sleep_duration"),
}
RATIO_COLS = list(RATIO_DEFS)
INTERACTION_DEFS = {
    "stress_sleep_quality": ("stress_level", "sleep_quality"),
    "activity_diet": ("physical_activity_level", "diet_type"),
    "smoking_activity": ("smoking_alcohol", "physical_activity_level"),
}

print({
    "python": platform.python_version(),
    "catboost": catboost.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
})


## 2. Data Loading

In [ ]:
def locate_data():
    roots = [Path("/kaggle/input"), Path("../input"), Path("../data/playground-series-s6e7"), Path("data/playground-series-s6e7"), Path(".")]
    for root in roots:
        if not root.exists():
            continue
        train_files = list(root.rglob("train.csv"))
        for train_path in train_files:
            test_path = train_path.with_name("test.csv")
            sample_path = train_path.with_name("sample_submission.csv")
            if test_path.exists() and sample_path.exists():
                return train_path, test_path, sample_path
    raise FileNotFoundError("train.csv, test.csv and sample_submission.csv were not found")

TRAIN_PATH, TEST_PATH, SAMPLE_PATH = locate_data()
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_PATH)

if DRY_RUN:
    fraction = DRY_TRAIN_ROWS / len(train)
    train = (
        train.groupby(TARGET, group_keys=False)
        .sample(frac=fraction, random_state=SEED)
        .reset_index(drop=True)
    )
    test = test.head(DRY_TEST_ROWS).copy()
    sample_submission = sample_submission.head(DRY_TEST_ROWS).copy()
    print("DRY_RUN active: results are pipeline validation only, not submission quality evidence.")

print("Data directory:", TRAIN_PATH.parent)
print("Train/Test:", train.shape, test.shape)
display(train.head())


In [ ]:
assert TARGET in train and TARGET not in test
assert train[ID_COL].is_unique and test[ID_COL].is_unique
assert set(train[TARGET].unique()) == set(CLASSES)
assert list(sample_submission.columns) == [ID_COL, TARGET]

schema = pd.DataFrame({
    "dtype": train.dtypes.astype(str),
    "missing_train": train.isna().sum(),
    "missing_test": test.isna().sum().reindex(train.columns),
    "nunique": train.nunique(dropna=False),
})
display(schema)
print("Duplicate train rows:", train.duplicated().sum())

### 2.1 Data Contract and Leakage Checks

In [ ]:
expected_features = set(NUM_COLS + CAT_COLS)
assert expected_features.issubset(train.columns)
assert expected_features.issubset(test.columns)
assert set(train.columns) - {TARGET} == set(test.columns)
assert not train[ID_COL].isin(test[ID_COL]).any(), "Train/test IDs overlap"

contract = pd.DataFrame({
    "train_dtype": train.dtypes.astype(str),
    "test_dtype": test.dtypes.astype(str).reindex(train.columns),
    "train_unique": train.nunique(dropna=False),
    "test_unique": test.nunique(dropna=False).reindex(train.columns),
})
display(contract)
print("Leakage checks passed: schemas align and IDs do not overlap.")

## 3. Lightweight Validation (EDA omitted for submission speed)


### 3.1 Target and Missing Values

In [ ]:
# Submission notebook: expensive EDA intentionally skipped.


### 3.2 Missingness–Target Signal

In [ ]:
# Submission notebook: expensive EDA intentionally skipped.


### 3.3 Train–Test Distribution Shift

In [ ]:
# Submission notebook: expensive EDA intentionally skipped.


### 3.4 Numerical Feature Distributions

In [ ]:
# Submission notebook: expensive EDA intentionally skipped.


### 3.5 Categorical Feature Distributions

In [ ]:
# Submission notebook: expensive EDA intentionally skipped.


### 3.6 Correlation Analysis

In [ ]:
# Submission notebook: expensive EDA intentionally skipped.


## 4. Outlier Analysis

In [ ]:
# Submission notebook: expensive EDA intentionally skipped.


### 4.1 Clipping and Removal Impact (Analysis Only)

In [ ]:
# Submission notebook: expensive EDA intentionally skipped.


## 5. Fold-Safe Feature Engineering

In [ ]:
class V2CorePreprocessor:
    def fit(self, frame):
        self.medians_ = frame[NUM_COLS].median()
        prepared = self._base(frame)
        outlier_sources = NUM_COLS + RATIO_COLS
        self.bounds_ = {
            col: tuple(prepared[col].quantile([0.005, 0.995])) for col in outlier_sources
        }
        flagged = self._flags(prepared)
        candidates = [f"{col}_outlier_{side}" for col in outlier_sources for side in ("low", "high")]
        self.active_flags_ = [c for c in candidates if flagged[c].nunique(dropna=False) > 1]

        categorical = CAT_COLS + list(INTERACTION_DEFS)
        self.category_levels_ = {
            col: sorted(flagged[col].astype(str).unique().tolist()) + ["__UNKNOWN__"]
            for col in categorical
        }
        transformed = self._finish(flagged)
        self.feature_names_ = transformed.columns.tolist()
        return self

    def transform(self, frame):
        flagged = self._flags(self._base(frame))
        return self._finish(flagged).reindex(columns=self.feature_names_)

    def fit_transform(self, frame):
        return self.fit(frame).transform(frame)

    def _base(self, frame):
        result = frame[NUM_COLS + CAT_COLS].copy()
        raw_missing = result.isna()
        for col in NUM_COLS + CAT_COLS:
            result[f"{col}_is_missing"] = raw_missing[col].astype("int8")
        result["missing_count"] = raw_missing.sum(axis=1).astype("int16")
        result[NUM_COLS] = result[NUM_COLS].fillna(self.medians_)
        result[CAT_COLS] = result[CAT_COLS].fillna("missing")

        for output, (numerator, denominator) in RATIO_DEFS.items():
            result[output] = result[numerator].div(result[denominator].add(1)).replace([np.inf, -np.inf], np.nan)
        for output, (left, right) in INTERACTION_DEFS.items():
            result[output] = result[left].astype(str) + "__" + result[right].astype(str)
        return result

    def _flags(self, result):
        result = result.copy()
        for col, (low, high) in self.bounds_.items():
            result[f"{col}_outlier_low"] = (result[col] < low).astype("int8")
            result[f"{col}_outlier_high"] = (result[col] > high).astype("int8")
        return result

    def _finish(self, result):
        result = result.copy()
        result["outlier_count"] = result[self.active_flags_].sum(axis=1).astype("int16")
        keep = (NUM_COLS + CAT_COLS +
                [f"{c}_is_missing" for c in NUM_COLS + CAT_COLS] +
                ["missing_count"] + RATIO_COLS + list(INTERACTION_DEFS) +
                self.active_flags_ + ["outlier_count"])
        result = result[keep]
        for col, levels in self.category_levels_.items():
            values = result[col].astype(str)
            result[col] = pd.Categorical(values.where(values.isin(levels), "__UNKNOWN__"), categories=levels)
        return result

In [ ]:
# Full preview intentionally skipped; each fold validates the transformed schema.
print("V2-Core preprocessor ready; fold-safe fitting will occur inside CV.")


## 6. Model Training and Validation

### 6.1 CatBoost GPU / CPU Selection


In [ ]:
def resolve_catboost_task_type():
    """Prefer Kaggle CUDA GPU and safely fall back to CPU."""
    probe_X = pd.DataFrame({"x": ["a", "b", "a", "b"], "n": [0., 1., 0., 1.]})
    probe_y = np.array([0, 1, 0, 1])
    try:
        probe = CatBoostClassifier(
            iterations=2, depth=2, loss_function="MultiClass",
            task_type="GPU", devices="0", verbose=False,
            allow_writing_files=False, random_seed=SEED,
        )
        probe.fit(probe_X, probe_y, cat_features=["x"])
        return "GPU"
    except Exception as error:
        print("GPU unavailable; falling back to CPU:", str(error).splitlines()[0])
        return "CPU"

TASK_TYPE = resolve_catboost_task_type()
print("Selected CatBoost task type:", TASK_TYPE)
if TASK_TYPE == "CPU":
    print("Kaggle notebook settings -> Accelerator -> GPU is recommended before running all cells.")


In [ ]:
MODEL_PARAMS = dict(
    loss_function="MultiClass",
    eval_metric="MultiClass",
    depth=6,
    learning_rate=0.035,
    iterations=100 if DRY_RUN else 5000,
    l2_leaf_reg=8.0,
    random_strength=1.0,
    bootstrap_type="Bayesian",
    bagging_temperature=0.5,
    od_type="Iter",
    od_wait=20 if DRY_RUN else 250,
    random_seed=SEED,
    allow_writing_files=False,
    verbose=20 if DRY_RUN else 100,
    task_type=TASK_TYPE,
)
if TASK_TYPE == "GPU":
    MODEL_PARAMS["devices"] = "0"
else:
    MODEL_PARAMS["thread_count"] = -1

X = train.drop(columns=[ID_COL, TARGET])
X_test = test.drop(columns=ID_COL)
y = train[TARGET].map(CLASS_TO_ID).to_numpy()

counts = np.bincount(y, minlength=len(CLASSES))
balanced = len(y) / (len(CLASSES) * counts)
sqrt_class_weights = np.sqrt(balanced)
sample_weight = sqrt_class_weights[y]
print("sqrt class weights:", dict(zip(CLASSES, sqrt_class_weights.round(6))))

cv = StratifiedKFold(n_splits=2 if DRY_RUN else 3, shuffle=True, random_state=SEED)
oof_proba = np.zeros((len(train), len(CLASSES)), dtype=np.float32)
test_proba = np.zeros((len(test), len(CLASSES)), dtype=np.float32)
fold_records, fold_importance = [], []

for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y), 1):
    started = time.perf_counter()
    processor = V2CorePreprocessor()
    X_tr = processor.fit_transform(X.iloc[train_idx])
    X_va = processor.transform(X.iloc[valid_idx])
    X_te = processor.transform(X_test)
    categorical = [c for c in X_tr.columns if str(X_tr[c].dtype) == "category"]
    for col in categorical:
        X_tr[col] = X_tr[col].astype(str)
        X_va[col] = X_va[col].astype(str)
        X_te[col] = X_te[col].astype(str)

    model = CatBoostClassifier(**MODEL_PARAMS)
    model.fit(
        X_tr, y[train_idx], sample_weight=sample_weight[train_idx],
        eval_set=(X_va, y[valid_idx]), cat_features=categorical,
    )
    valid_proba = model.predict_proba(X_va)
    fold_test_proba = model.predict_proba(X_te)
    valid_proba /= valid_proba.sum(axis=1, keepdims=True)
    fold_test_proba /= fold_test_proba.sum(axis=1, keepdims=True)
    oof_proba[valid_idx] = valid_proba
    test_proba += fold_test_proba.astype(np.float32) / cv.n_splits

    valid_pred = valid_proba.argmax(1)
    recalls = recall_score(y[valid_idx], valid_pred, labels=range(3), average=None, zero_division=0)
    record = {
        "fold": fold,
        "balanced_accuracy": balanced_accuracy_score(y[valid_idx], valid_pred),
        "best_iteration": model.get_best_iteration() + 1,
        "feature_count": X_tr.shape[1],
        "minutes": (time.perf_counter() - started) / 60,
        **{f"recall_{label}": recalls[i] for i, label in enumerate(CLASSES)},
    }
    fold_records.append(record)
    fold_importance.append(pd.DataFrame({
        "feature": X_tr.columns, "gain": model.get_feature_importance(), "fold": fold,
    }))
    np.save(f"oof_proba_fold{fold}.npy", valid_proba.astype(np.float32))
    np.save(f"test_proba_fold{fold}.npy", fold_test_proba.astype(np.float32))
    print(f"Fold {fold}: balanced accuracy={record['balanced_accuracy']:.6f}, best_iteration={record['best_iteration']}, minutes={record['minutes']:.2f}")
    del X_tr, X_va, X_te, model, processor, valid_proba, fold_test_proba
    gc.collect()

fold_metrics = pd.DataFrame(fold_records)
print(f"Raw CV: {fold_metrics.balanced_accuracy.mean():.6f} ± {fold_metrics.balanced_accuracy.std(ddof=1):.6f}")
display(fold_metrics.style.format({c: "{:.6f}" for c in fold_metrics if "accuracy" in c or "recall" in c}))
np.save("oof_proba.npy", oof_proba)
np.save("test_proba.npy", test_proba)


## 7. Evaluation

In [ ]:
raw_pred = oof_proba.argmax(axis=1)
raw_report = pd.DataFrame(classification_report(y, raw_pred, target_names=CLASSES, output_dict=True, zero_division=0)).T
raw_cm = confusion_matrix(y, raw_pred, normalize="true")
print("Raw OOF balanced accuracy:", balanced_accuracy_score(y, raw_pred))
display(raw_report.style.format("{:.4f}"))

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
ConfusionMatrixDisplay(raw_cm, display_labels=CLASSES).plot(cmap="Blues", values_format=".3f", ax=axes[0], colorbar=False)
axes[0].set_title("Normalized OOF confusion matrix")

importance = (pd.concat(fold_importance).groupby("feature", as_index=False)["gain"].mean()
              .sort_values("gain", ascending=False).head(20))
sns.barplot(data=importance, y="feature", x="gain", ax=axes[1], color="#4C72B0")
axes[1].set_title("Top-20 mean gain importance")

metric_cols = ["balanced_accuracy"] + [f"recall_{label}" for label in CLASSES]
fold_long = fold_metrics.melt(id_vars="fold", value_vars=metric_cols, var_name="metric", value_name="score")
sns.barplot(data=fold_long, x="metric", y="score", hue="fold", ax=axes[2], palette="viridis")
axes[2].set_ylim(0.75, 1.0)
axes[2].set_title("Fold metrics")
axes[2].tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

fold_metrics.to_csv("fold_metrics.csv", index=False)
importance.to_csv("feature_importance_top20.csv", index=False)

## 8. OOF-only Class-Multiplier Tuning


In [ ]:
# CatBoost is tuned on its own OOF probabilities; public LB is never used here.
scale_values = np.arange(0.98, 1.0201, 0.01) if DRY_RUN else np.arange(0.95, 1.0501, 0.005)
best_score = -1.0
MULTIPLIERS = np.ones(3)
search_rows = []
for fit_scale in scale_values:
    for unhealthy_scale in scale_values:
        candidate = np.array([1.0, fit_scale, unhealthy_scale])
        pred = np.argmax(oof_proba * candidate, axis=1)
        score = balanced_accuracy_score(y, pred)
        search_rows.append({"fit_scale": fit_scale, "unhealthy_scale": unhealthy_scale, "balanced_accuracy": score})
        if score > best_score:
            best_score = score
            MULTIPLIERS = candidate

tuned_pred = np.argmax(oof_proba * MULTIPLIERS, axis=1)
tuned_score = balanced_accuracy_score(y, tuned_pred)
tuned_report = pd.DataFrame(classification_report(y, tuned_pred, target_names=CLASSES, output_dict=True, zero_division=0)).T
print("Selected multipliers:", MULTIPLIERS)
print("Tuned OOF balanced accuracy:", tuned_score)
display(tuned_report.style.format("{:.4f}"))

search = pd.DataFrame(search_rows)
pivot = search.pivot(index="fit_scale", columns="unhealthy_scale", values="balanced_accuracy")
plt.figure(figsize=(9, 7))
sns.heatmap(pivot, cmap="viridis")
plt.title("CatBoost OOF multiplier search")
plt.show()

with open("best_multipliers.json", "w") as handle:
    json.dump(dict(zip(CLASSES, MULTIPLIERS.tolist())), handle, indent=2)


## 9. Create `submission.csv`

In [ ]:
test_pred = np.argmax(test_proba * MULTIPLIERS, axis=1)
submission = sample_submission[[ID_COL]].copy()
submission[TARGET] = np.array(CLASSES)[test_pred]

assert submission.shape == sample_submission.shape
assert submission[ID_COL].equals(sample_submission[ID_COL])
assert submission[TARGET].isin(CLASSES).all()
assert submission[TARGET].notna().all()

submission_distribution = submission[TARGET].value_counts().reindex(CLASSES, fill_value=0)
display(pd.DataFrame({
    "count": submission_distribution,
    "rate": submission_distribution / len(submission),
}))

output_name = "submission_dry_run.csv" if DRY_RUN else "submission.csv"
submission.to_csv(output_name, index=False)
fold_metrics.to_csv("fold_metrics.csv", index=False)
print(f"Saved: {output_name}", submission.shape)
print("Submit only after reviewing OOF score, fold stability and class distribution.")
display(submission.head())
